# Data Scaling and Working with Dataframes

This notebook completes the exercise using `calif_housing_data.csv`.
I am working through normalization, standardization, and dataframe tasks step by step.

In [1]:
import numpy as np
import pandas as pd

## 1) Data Normalization (0 to 1)

The goal is to rescale a vector so its smallest value becomes 0 and its largest becomes 1.  
This is done using min–max normalization.

In [16]:
def normalize_vector(x):
        # Ensure numeric array
    x = np.asarray(x, dtype=float)

    # min-max scaling [0-1]
    x_min = np.min(x)
    x_max = np.max(x)
    denom = x_max - x_min
    
    # If all values are the same, avoid dividing by zero
    if denom == 0:
        return np.zeros_like(x)
    
    return (x - x_min) / denom

# test
test = np.array([10, 12, 15, 20])
normalize_vector(test)

array([0. , 0.2, 0.5, 1. ])

In [31]:
# apply normalization to test data 

norm = normalize_vector(test)

# verify output is within the min/ max range [0-1]

norm.min(), norm.max()

(np.float64(0.0), np.float64(1.0))

## 2) Data Standardization (z-score)

Goal: convert a vector to z-scores:

x_std = (x - mean(x)) / std(x)

This should produce a result with mean near 0 and std near 1 (if std is not 0).

In [17]:
def standardize_vector(x):
    # ensure numeric array
    x = np.asarray(x, dtype=float)

    # compute the center and spread
    mean = np.mean(x)
    std = np.std(x, ddof=1)  # sample standard deviation
    
    if std == 0:     # avoid division by zero 
        return np.zeros_like(x)
    # applying the z-score transformation
    return (x - mean) / std

# test
standardize_vector(test)

array([-0.97716212, -0.51732112,  0.17244037,  1.32204287])

In [5]:
z = standardize_vector(test)
z.mean(), z.std(ddof=1)

(np.float64(5.551115123125783e-17), np.float64(1.0))

## 3) Working with a DataFrame

Now I will load `calif_housing_data.csv` and answer questions (a) through (e).

In [28]:
df = pd.read_csv("calif_housing_data.csv")

df.head()
# loaded housing data and checks the return

,housing_median_age,total_bedrooms,households,median_income,median_house_value
0,41,129.0,126,8.3252,452600.0
1,21,1106.0,1138,8.3014,358500.0
2,52,190.0,177,7.2574,352100.0
3,52,235.0,219,5.6431,341300.0
4,52,280.0,259,3.8462,342200.0


In [7]:
df.columns

Index(['housing_median_age', 'total_bedrooms', 'households', 'median_income',
       'median_house_value'],
      dtype='object')

### (a) Row count

I check for how many observations are in the dataset.

In [27]:
df.shape[0]
# number of observations

20640

### (b) Target vector

The goal is to predict median house value, so the target vector is the `median_house_value` column.

In [25]:
# target the variable: median house value
y = df["median_house_value"]

y.head()

0    452600.0
1    358500.0
2    352100.0
3    341300.0
4    342200.0
Name: median_house_value, dtype: float64

### (c) New feature: bedrooms per household

I create a new feature by dividing total bedrooms by the number of households.  
This gives the average number of bedrooms per household for each observation.

In [23]:
# finding the average number of bedreooms per household
df["bedrooms_per_household"] = df["total_bedrooms"] / df["households"]

df[["total_bedrooms", "households", "bedrooms_per_household"]].head()

,total_bedrooms,households,bedrooms_per_household
0,129.0,126,1.023810
1,1106.0,1138,0.971880
2,190.0,177,1.073446
3,235.0,219,1.073059
4,280.0,259,1.081081


### (d) New dataframe with three features

I will keep:
- housing_median_age
- median_income
- bedrooms_per_household

In [22]:
# feature set for modeling
X = df[["housing_median_age", "median_income", "bedrooms_per_household"]].copy()

X.head()

,housing_median_age,median_income,bedrooms_per_household
0,41,8.3252,1.023810
1,21,8.3014,0.971880
2,52,7.2574,1.073446
3,52,5.6431,1.073059
4,52,3.8462,1.081081


### (e) Standardize the features

Here, I standardize each feature in `X` using z-scores.  
This centers each column at 0 and scales it to a standard deviation of 1 so the features are on a comparable scale.

In [21]:
# standardize features to z-scores
X_standardized = (X - X.mean()) / X.std(ddof=1)

X_standardized.head()

,housing_median_age,median_income,bedrooms_per_household
0,0.982119,2.344709,-0.153859
1,-0.607004,2.332181,-0.262930
2,1.856137,1.782656,-0.049603
3,1.856137,0.932945,-0.050416
4,1.856137,-0.012881,-0.033567


In [19]:
# check that mean is near 0 and std = 1
pd.DataFrame({
   
    "mean": X_standardized.mean(),
    "std_ddof1": X_standardized.std(ddof=1)
})

,mean,std_ddof1
housing_median_age,3.855658e-17,1.0
median_income,7.711317e-17,1.0
bedrooms_per_household,9.180408e-17,1.0


## Final verification

After standardizing the feature matrix, each variable has a mean approximately equal to 0 and a standard deviation of 1, confirming that the z-score transformation was applied correctly using the sample standard deviation.

Standardization converts each observation into a measure of how many standard deviations it is from the mean. This places all features on a common scale, allowing for fair comparison and preventing variables with larger numeric ranges from having an outsized influence in analysis or modeling.